# 📊 Telco Customer Churn Prediction
**Goal:** Predict which customers are likely to churn using an ensemble ML pipeline.

---
## Pipeline Overview
1. Imports & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Encoding, Train/Test Split & SMOTE
6. Baseline Model (Random Forest + GridSearchCV)
7. Bayesian Hyperparameter Optimization (Optuna)
8. Final Ensemble Model (RF + LR + XGBoost)
9. Evaluation & Visualizations

---
## Step 1: Imports & Data Loading

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.xls')
print(f"Dataset shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/WA_Fn-UseC_-Telco-Customer-Churn.xls'

---
## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print("=== Dataset Info ===")
display(df.info())

print("\n=== Missing Values ===")
display(df.isnull().sum())

print("\n=== Class Distribution ===")
display(df['Churn'].value_counts())

In [ ]:
# Churn Distribution by Tenure Group
def bin_tenure(tenure):
    if tenure <= 12:
        return '0-12_Months'
    elif tenure <= 24:
        return '12-24_Months'
    elif tenure <= 48:
        return '24-48_Months'
    elif tenure <= 60:
        return '48-60_Months'
    else:
        return 'Over_60_Months'

df['tenure_group'] = df['tenure'].apply(bin_tenure)

plt.figure(figsize=(10, 5))
sns.countplot(
    data=df,
    x='tenure_group',
    hue='Churn',
    palette='magma',
    order=['0-12_Months', '12-24_Months', '24-48_Months', '48-60_Months', 'Over_60_Months']
)
plt.title('Churn Distribution by Tenure Group')
plt.xlabel('Tenure Group')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## Step 3: Data Cleaning & Preprocessing

In [ ]:
# Fix TotalCharges (stored as string in raw data), drop ID column, encode target
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df.drop(columns=['customerID', 'tenure_group'], inplace=True)  # remove helper EDA column too
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Cleaned dataset shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean():.2%}")

---
## Step 4: Feature Engineering

In [ ]:
# Tenure group (ordinal bins)
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4+yr']
)

# Average spend rate (avoids raw TotalCharges being misleading for short-tenure customers)
df['ChargesPerMonth'] = df['TotalCharges'] / (df['tenure'] + 1)

# High-risk flag: month-to-month contract + Fiber optic (highest churn combo)
df['HighRisk'] = (
    (df['Contract'] == 'Month-to-month') &
    (df['InternetService'] == 'Fiber optic')
).astype(int)

print("New features added: TenureGroup, ChargesPerMonth, HighRisk")
df[['tenure', 'TenureGroup', 'ChargesPerMonth', 'HighRisk']].head()

---
## Step 5: Encoding, Train/Test Split & SMOTE

In [ ]:
# One-hot encode all categorical columns
df_final = pd.get_dummies(df, drop_first=True)

X_f = df_final.drop('Churn', axis=1)
y_f = df_final['Churn']

# Stratified train/test split
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_f, y_f, test_size=0.2, random_state=42, stratify=y_f
)

# Apply SMOTE only on training data to handle class imbalance
smote = SMOTE(random_state=42)
X_res_f, y_res_f = smote.fit_resample(X_train_f, y_train_f)

# Scale numerical features (important for Logistic Regression in the ensemble)
scaler = StandardScaler()
X_res_scaled = scaler.fit_transform(X_res_f)
X_test_scaled = scaler.transform(X_test_f)

print(f"Training set after SMOTE: {X_res_f.shape}")
print(f"Test set: {X_test_f.shape}")

---
## Step 6: Baseline Model — Random Forest + GridSearchCV

In [ ]:
# Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    verbose=1,
    n_jobs=-1
)
grid_search.fit(X_res_f, y_res_f)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best F1 Score (CV): {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate best GridSearch model on test set
best_rf = grid_search.best_estimator_
y_pred_grid = best_rf.predict(X_test_f)

print("Baseline (GridSearchCV) Classification Report:")
print(classification_report(y_test_f, y_pred_grid))

---
## Step 7: Bayesian Hyperparameter Optimization (Optuna)

In [ ]:
# Install Optuna if not already available
# !pip install optuna -q
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    }
    clf = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    score = cross_val_score(clf, X_res_f, y_res_f, cv=3, scoring='f1').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Best Optuna Params: {study.best_params}")
print(f"Best F1 Score: {study.best_value:.4f}")

In [ ]:
# Train and evaluate the Optuna-optimized Random Forest
optuna_rf = RandomForestClassifier(**study.best_params, random_state=42)
optuna_rf.fit(X_res_f, y_res_f)

y_pred_opt = optuna_rf.predict(X_test_f)
print("Classification Report: Bayesian Optimized Random Forest")
print(classification_report(y_test_f, y_pred_opt))

---
## Step 8: Final Ensemble Model (RF + Logistic Regression + XGBoost)

In [ ]:
# Define individual models
xgb_best = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, eval_metric='logloss', random_state=42
)
rf_best = RandomForestClassifier(
    n_estimators=199, max_depth=27, min_samples_split=3, random_state=42
)
lr_best = LogisticRegression(max_iter=2000, class_weight='balanced')

# Soft-voting ensemble (uses predicted probabilities)
ensemble = VotingClassifier(
    estimators=[('rf', rf_best), ('lr', lr_best), ('xgb', xgb_best)],
    voting='soft'
)
ensemble.fit(X_res_scaled, y_res_f)

# Predict
y_pred_ensemble = ensemble.predict(X_test_scaled)
y_proba_ensemble = ensemble.predict_proba(X_test_scaled)[:, 1]

print("--- Final Ensemble Pipeline Performance ---")
print(classification_report(y_test_f, y_pred_ensemble))
print(f"Final ROC-AUC: {roc_auc_score(y_test_f, y_proba_ensemble):.4f}")

---
## Step 9: Evaluation & Visualizations

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_f, y_pred_ensemble)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=ax, cmap='YlGnBu')
ax.set_title('Confusion Matrix: Final Ensemble Model')
plt.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 Feature Importances from Optuna-tuned Random Forest
importances = pd.Series(
    optuna_rf.feature_importances_, index=X_f.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(15).plot(kind='barh', color='teal')
plt.title('Top 15 Feature Importances (Optuna-Optimized RF)')
plt.gca().invert_yaxis()
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()